In [1]:
import os
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

from RLAlg.utils import set_seed_everywhere

from model.encoder import FrameObservationEncoderNet, EncoderNet
from state_frame_dataset import get_dataloader

In [2]:
class Alignment(nn.Module):
    def __init__(self, vector_weight):
        super().__init__()
        
        self.frame_encoder = FrameObservationEncoderNet(6, 128)

        encoder = EncoderNet(39, [128, 128, 128, 128, 128])

        encoder.load_state_dict(vector_weight)

        self.vector_encoder = encoder.layers[:]

        for param in self.vector_encoder.parameters():
            param.requires_grad = False

    def encoder_vectors(self, vectors):
        vector_features = vectors
        with torch.no_grad():
            for layer in self.vector_encoder:
                vector_features = layer(vector_features)

        return vector_features

    def encode_frames(self, frames):
        frame_features = self.frame_encoder(frames)

        return frame_features
    
    def forward(self, frames, vectors):
        frame_features = self.encode_frames(frames)
        vector_features = self.encoder_vectors(vectors)

        return frame_features, vector_features

In [3]:
weight_folder = f"weights/ddpg/pickplace"
        
if not os.path.exists(weight_folder):
    os.makedirs(weight_folder)

set_seed_everywhere(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder_weight, _, _ = torch.load(f"{weight_folder}/actor_0_100.pt", weights_only=True)
model = Alignment(encoder_weight).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer=optimizer, T_max=10)
dataloader = get_dataloader("pickplace")
size = len(dataloader.dataset)

In [4]:
def cosine_alignment_loss(x, y):
    x = F.normalize(x, dim=-1)
    y = F.normalize(y, dim=-1)
    return 1 - (x * y).sum(dim=-1).mean()

In [5]:
for _ in range(10):
    start_time = time.time()
    running_loss = 0.0
    for _, (vectors, frames) in enumerate(dataloader):
        vectors = vectors.to(device)
        frames = frames.to(device)
        
        frame_features, vector_features = model(frames, vectors)
        
        loss = cosine_alignment_loss(frame_features, vector_features)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * vectors.size(0)
    
    scheduler.step()
    end_time = time.time()
    train_loss = running_loss / size
    print(f"Train Loss: {train_loss:.4f}")
    elapsed_time = end_time - start_time
    print(f"The loop took {elapsed_time:.2f} seconds to complete.")

torch.save(model.frame_encoder.state_dict(), f"weights/ddpg/pickplace/frame_encoder_0_mse.pt")

Train Loss: 0.1285
The loop took 74.26 seconds to complete.
Train Loss: 0.0696
The loop took 56.97 seconds to complete.
Train Loss: 0.0608
The loop took 56.99 seconds to complete.
Train Loss: 0.0542
The loop took 56.94 seconds to complete.
Train Loss: 0.0468
The loop took 57.70 seconds to complete.
Train Loss: 0.0398
The loop took 57.03 seconds to complete.
Train Loss: 0.0320
The loop took 57.20 seconds to complete.
Train Loss: 0.0243
The loop took 57.62 seconds to complete.
Train Loss: 0.0172
The loop took 57.51 seconds to complete.
Train Loss: 0.0121
The loop took 57.28 seconds to complete.


In [6]:
frame_features

tensor([[ 0.0743, -0.0467,  0.1435,  ..., -0.0108,  0.1642, -0.0502],
        [ 0.1703, -0.0448,  0.0253,  ...,  0.0594,  0.1369, -0.0380],
        [-0.0373,  0.1915, -0.0295,  ..., -0.0426, -0.0477,  0.0292],
        ...,
        [-0.0342,  0.1409, -0.0431,  ..., -0.0502, -0.0356, -0.0364],
        [ 0.0582, -0.0317, -0.0418,  ..., -0.0179, -0.0275, -0.0548],
        [-0.0404,  0.0844, -0.0408,  ..., -0.0409, -0.0487, -0.0451]],
       device='cuda:0', grad_fn=<TanhBackward0>)

In [7]:
vector_features

tensor([[ 0.4734, -0.2773,  0.7579,  ..., -0.0622,  0.8866, -0.2754],
        [ 1.0861, -0.2721, -0.0082,  ...,  0.0982,  1.0415, -0.1575],
        [-0.2490,  1.1555, -0.2618,  ..., -0.2785, -0.2784,  0.1638],
        ...,
        [-0.2572,  0.8690, -0.2606,  ..., -0.2785, -0.2518, -0.2118],
        [ 0.1055, -0.2231, -0.2614,  ..., -0.1525, -0.1345, -0.2588],
        [-0.2697,  0.4401, -0.2721,  ..., -0.2486, -0.2785, -0.2645]],
       device='cuda:0')